Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [ ]:
NO_RECORDS = 1000

In [114]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests

In [115]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [116]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [117]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [118]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y', 'title': 'Japan (City of Joso Flood)', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><header status=""><identifier>doi:10.34894/YQ0L4Y</identifier><datestamp>2025-12-01T01:00:23Z</datestamp><setSpec>TUDelft</setSpec><setSpec>dataversenl</setSpec></header><metadata><resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"><identifier identifierType="DOI">10.34894/YQ0L4Y</identifier><creators><creator><creatorName nameType="Personal">N/A</creatorName></creator></creators><titles><title>Japan (City of Joso Flood)</title></titles><publisher>DataverseNL</publisher><publicationYear>2016</publicationYear><dates><date dateType="Submitted">2015-11-09</date><date dateType="Updated">2015-11-26</d

In [119]:
record_titles

['Japan (City of Joso Flood)',
 'Italian Guilds Database',
 'EU Policy Briefing on Whole of Society Conflict Prevention and Peacebuilding',
 'Lab journal',
 'Uses of Information and Communication Technologies (ICTs) for EU Conflict Prevention and Peacebuilding',
 'Sensitivity and validity of psychometric tests for assessing driving impairment: effects of sleep deprivation',
 'Civil-Military Synergy at Operational Level in EU External Action',
 'Strong, but wrong: Lay people’s and police officers’ beliefs about verbal and nonverbal cues to deception',
 'Dataset_Bamelis_et_al_2015',
 'Survey among Swiss Abroad',
 'Data sharing and re-use: Researcher practices, attitudes and needs - 2016',
 'Befragung junger deutschschweizer Familien zur Bedeutung von religiösen Ritualen - 2005',
 'Interest Group Preferences in Deficit Countries – Ireland, Spain and Greece',
 'ÜGK / COFO / VECOF 2023, Languages Grade 11: Keys for Data Linkage',
 'Umfrage: Immigrantinnen und Immigranten in der Schweiz-2013

In [120]:
record_raw_metadata[10]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:oai="http://www.openarchives.org/OAI/2.0/" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>oai:swissubase.ch:b70e44ac-73e0-454d-a056-7a619b71b811</identifier>\n    <datestamp>2026-03-24T12:04:32Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.23662/FORS-DS-849-1</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Personal">Team FORS, Archive</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title xml:lang="en">Data sharing and re-use: Researcher practices, attitudes and needs - 2016</da

In [121]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [122]:
raw_xml = ET_from_string(record_raw_metadata[13])

In [123]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x29099ebbdc0>

In [124]:
#print(ET.tostring(raw_xml, pretty_print=True).decode())

In [125]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['ÜGK / COFO / VECOF 2023, Languages Grade 11: Keys for Data Linkage']

In [126]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://datacite.org/schema/kernel-4}title at 0x2909a0f30c0>]
ÜGK / COFO / VECOF 2023, Languages Grade 11: Keys for Data Linkage


In [127]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element
#element[0].text

[<Element {http://datacite.org/schema/kernel-4}identifier at 0x2909a0f27c0>]

In [128]:
raw_xml.xpath("//*[local-name()='date']/text()")

['2025-07-14']

In [129]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

# xpath returns a list of strings if /text() is included

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

['10.48573/yp7t-tn85']

In [130]:
record_number = 6

print(ET.tostring(ET_from_string(record_raw_metadata[record_number - 1]), pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header status="">
    <identifier>doi:10.34894/17AUY3</identifier>
    <datestamp>2025-12-01T01:00:23Z</datestamp>
    <setSpec>MaastrichtUniversity</setSpec>
    <setSpec>dataversenl</setSpec>
  </header>
  <metadata>
    <resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">
      <identifier identifierType="DOI">10.34894/17AUY3</identifier>
      <creators>
        <creator>
          <creatorName nameType="Personal">Jongen, Stefan</creatorName>
          <givenName>Stefan</givenName>
          <familyName>Jongen</familyName>
          <affiliation>Maastricht University</affiliation>
        </creator>
        <creator>
          <creatorName nameType="Personal">Perrier, Joy</creatorName>
          <givenName>

In [131]:
rec = ET_from_string(record_raw_metadata[5])

In [132]:
rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")[0]

'10.34894/17AUY3'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [133]:
def openaire_doi_get_info(doi):

    # If the DOI starts with http(s) strip the start
    doi_param = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")

    response = requests.get(
        "https://api.openaire.eu/graph/v2/researchProducts",
        params={"pid": doi_param}
    )
    data = response.json()
    results = data.get("results", [])
    return results

In [134]:
rec = ET_from_string(record_raw_metadata[9])
doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

if doi == []:
    print("NO DOI")
else:
    print(openaire_doi_get_info(doi[0]))

DEBUG:    2026-05-21 09:22:11 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 09:22:11 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.23662%2FFORS-DS-690-1 HTTP/1.1" 200 None
[{'authors': [{'fullName': 'Blatter, Joachim', 'name': 'Joachim', 'surname': 'Blatter', 'rank': 1, 'pid': None}, {'fullName': 'Schlenker, Andrea', 'name': 'Andrea', 'surname': 'Schlenker', 'rank': 2, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': None, 'mainTitle': 'Survey among Swiss Abroad', 'subTitle': None, 'descriptions': ['This project is a prolongation of our research project on political attitudes and behaviour of dual citizens in Switzerland („Dual Citizens: Hazard or Vanguard of Citizenship in the (Post-)Westphalian Order?“, SNF project number: 137693, November 2011 to February 2015).

In [135]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 148, 149, 150, 151, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 2

In [ ]:
import random
import time

results = []
for i in ix_has_doi:
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

    if doi == []:
        print(f"{i} : NO DOI")
    else:
        doi = doi[0]

        # small pause to not overwhelm API
        time.sleep(random.uniform(0.01, 1))

        print(f"DOI for record {i}")
        res = openaire_doi_get_info(doi)
        print(res)
        results.append(res)

DOI for record 0
DEBUG:    2026-05-21 09:28:33 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 09:28:34 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.34894%2FYQ0L4Y HTTP/1.1" 200 None
[{'authors': [{'fullName': 'N/A', 'name': None, 'surname': None, 'rank': 1, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': [{'code': 'US', 'label': 'United States', 'provenance': None}, {'code': 'NL', 'label': 'Netherlands', 'provenance': None}], 'subjects': None, 'mainTitle': 'Japan (City of Joso Flood)', 'subTitle': None, 'descriptions': ['N/A'], 'publicationDate': '2016-01-01', 'publisher': 'DataverseNL', 'embargoEndDate': '2016-11-29', 'sources': None, 'formats': ['application/octet-stream', 'application/pdf', 'text/plain; charset=US-ASCII', 'text/plain', 'charset=US-ASCII'], 'contributors': None, 

In [148]:
record_raw_metadata[5]


'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><header status=""><identifier>doi:10.34894/17AUY3</identifier><datestamp>2025-12-01T01:00:23Z</datestamp><setSpec>MaastrichtUniversity</setSpec><setSpec>dataversenl</setSpec></header><metadata><resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"><identifier identifierType="DOI">10.34894/17AUY3</identifier><creators><creator><creatorName nameType="Personal">Jongen, Stefan</creatorName><givenName>Stefan</givenName><familyName>Jongen</familyName><affiliation>Maastricht University</affiliation></creator><creator><creatorName nameType="Personal">Perrier, Joy</creatorName><givenName>Joy</givenName><familyName>Perrier</familyName><affiliation>INSERM, U 1075 COMETE, Caen, 14032, France</affiliation></creator><creator><crea

In [157]:
results[6]

[{'authors': [{'fullName': 'Shyamika Jayasundara-Smits',
    'name': None,
    'surname': None,
    'rank': 1,
    'pid': None}],
  'openAccessColor': None,
  'publiclyFunded': None,
  'type': 'dataset',
  'language': {'code': 'und', 'label': 'Undetermined'},
  'countries': [{'code': 'NL', 'label': 'Netherlands', 'provenance': None},
   {'code': 'US', 'label': 'United States', 'provenance': None}],
  'subjects': [{'subject': {'scheme': 'SDG', 'value': '16. Peace & justice'},
    'provenance': None}],
  'mainTitle': 'Civil-Military Synergy at Operational Level in EU External Action',
  'subTitle': None,
  'descriptions': ['This report was produced as part of the project “Whole of Society Conflict Prevention and Peacebuilding”. It reflects on the challenges found in trying to enhance civil-military synergies in EU action, and identifies opportunities based on the experiences of practitioners on good and bad practices at the operational level. It also puts forward several key recommendati

In [156]:
results[5][0]['subjects']

In [ ]:
# standardised answers, they always have Subjects although it's mostly empty
# extract them, see if any are non - empty

In [158]:
result_subjects = []
for rez in results:
    subj = rez[0]['subjects']
    result_subjects.append(subj)

In [159]:
result_subjects

[None,
 None,
 None,
 None,
 None,
 None,
 [{'subject': {'scheme': 'SDG', 'value': '16. Peace & justice'},
   'provenance': None}],
 None,
 None,
 None]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword